# Layer-Wise Interference Diagnostics for Task Vector Merging

This notebook localizes layer-level interference between IF and Math task vectors using:

- `norm_if_l = ||Δ_if_l||_2`
- `norm_math_l = ||Δ_math_l||_2`
- `cos_l = <Δ_if_l, Δ_math_l> / (||Δ_if_l|| ||Δ_math_l||)`
- `conflict_l(τ) = mean[|Δ_if| > τ, |Δ_math| > τ, sign mismatch]`
- layer-wise sign conflict statistics and visual diagnostics
- `nonzero_overlap_ratio = |supp(Δ_if) ∩ supp(Δ_math)| / N`
- `nonzero_overlap_jaccard = |supp(Δ_if) ∩ supp(Δ_math)| / |supp(Δ_if) ∪ supp(Δ_math)|`
- `sign_conflict_ratio_nonzero = mean[sign mismatch | Δ_if != 0, Δ_math != 0]`

Where:
- `Δ_if = θ_if - θ_base`
- `Δ_math = θ_math - θ_base`


## Normalization Decision

For diagnostics, this notebook intentionally reports **both**:

1. **Raw norms** (`||Δ||_2`): reveal absolute update energy (how hard a layer moved).
2. **Normalized shares** (`||Δ_if|| / (||Δ_if|| + ||Δ_math||)`): reveal task dominance independent of absolute scale.

This resolves the common ambiguity around whether gradients/task vectors should be normalized. We should not throw away raw scale, but we should also add normalized views for fair cross-layer comparison.


In [ ]:
from __future__ import annotations

import gc
import json
import math
import re
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, Tuple

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM

# --------------------------------------------------------------------------------------
# Experiment configuration
# --------------------------------------------------------------------------------------
BASE_MODEL_ID = "Qwen/Qwen3-1.7B"
IF_MODEL_PATH = Path("/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-ifrl_ifeval/global_step_50/actor/huggingface")
MATH_MODEL_PATH = Path("/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-math/stage2/global_step_40/actor/huggingface")

# Tau is layer-adaptive via RMS scaling: tau_l = tau_rms_factor * 0.5 * (rms_if_l + rms_math_l)
TAU_RMS_FACTOR = 0.1

# Include special non-block parameters that are relevant to output behavior.
# - layer_embed: always included
# - layer_lm_head: included when it exists as an explicit parameter
# Explicitly exclude final norm to avoid degenerate interpretation cases.
INCLUDE_LAYER_EMBED = True
INCLUDE_LAYER_LM_HEAD = True
EXCLUDE_LAYER_FINAL_NORM = True

ARTIFACT_DIR = Path("merging_analysis/artifacts/layer_interference")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

for required_path in [IF_MODEL_PATH, MATH_MODEL_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(f"Required checkpoint path does not exist: {required_path}")

sns.set_theme(style="whitegrid", context="talk")
print(f"Artifacts will be written to: {ARTIFACT_DIR}")


In [ ]:
def parse_layer_name(param_name: str) -> str:
    """Map parameter names to stable layer identifiers.

    Args:
        param_name: Full parameter name from `model.named_parameters()`.

    Returns:
        Layer identifier string. Decoder blocks are encoded as `layer_XX`.
    """

    match = re.search(r"model\.layers\.(\d+)\.", param_name)
    if match:
        layer_idx = int(match.group(1))
        return f"layer_{layer_idx:02d}"

    # Non-block parameters are tracked explicitly to avoid dropping potential signals.
    if param_name.startswith("model.embed_tokens"):
        return "layer_embed"
    if param_name.startswith("model.norm"):
        return "layer_final_norm"
    if param_name.startswith("lm_head"):
        return "layer_lm_head"
    return "layer_other"


def should_include_parameter_in_analysis(param_name: str, param_tensor: torch.Tensor) -> bool:
    """Return whether a parameter should be included in layer interference analysis.

    Design rationale:
    - Keep all decoder block parameters (`model.layers.*`) because they dominate task-vector behavior.
    - Keep `layer_embed` because token-space alignment can affect generation behavior.
    - Keep `layer_lm_head` when present to capture output-projection interference.
    - Drop `layer_final_norm` by default because it often has near-degenerate updates,
      which can create misleading edge-case metrics.

    Args:
        param_name: Parameter name.
        param_tensor: Parameter tensor.

    Returns:
        Boolean inclusion flag.
    """

    if not torch.is_floating_point(param_tensor):
        return False

    if param_name.startswith("model.layers."):
        return True

    if INCLUDE_LAYER_EMBED and param_name.startswith("model.embed_tokens"):
        return True

    if INCLUDE_LAYER_LM_HEAD and param_name.startswith("lm_head"):
        return True

    if EXCLUDE_LAYER_FINAL_NORM and param_name.startswith("model.norm"):
        return False

    return False


def layer_sort_key(layer_name: str) -> Tuple[int, str]:
    """Return sortable key that keeps decoder layers first and ordered numerically."""

    match = re.fullmatch(r"layer_(\d+)", layer_name)
    if match:
        return (0, int(match.group(1)))

    # Keep special layers after decoder stack in a stable order.
    special_order = {
        "layer_embed": 0,
        "layer_lm_head": 1,
        "layer_final_norm": 2,
        "layer_other": 3,
    }
    return (1, special_order.get(layer_name, 99), layer_name)


def load_causal_lm(model_name_or_path: str | Path, dtype: torch.dtype = torch.float16):
    """Load a model checkpoint on CPU for offline parameter analysis.

    Args:
        model_name_or_path: HF model ID or local checkpoint directory.
        dtype: Load dtype for model weights.

    Returns:
        `AutoModelForCausalLM` instance on CPU.
    """

    model = AutoModelForCausalLM.from_pretrained(
        str(model_name_or_path),
        device_map="cpu",
        low_cpu_mem_usage=True,
        torch_dtype=dtype,
        trust_remote_code=True,
    )
    return model


@dataclass
class Pass1Accumulator:
    """Accumulator used during first pass to estimate layer-wise scale and alignment."""

    sum_sq_if: float = 0.0
    sum_sq_math: float = 0.0
    dot_sum: float = 0.0
    numel: int = 0


In [ ]:
def collect_pass1_stats(
    base_model: AutoModelForCausalLM,
    if_model: AutoModelForCausalLM,
    math_model: AutoModelForCausalLM,
) -> Dict[str, Pass1Accumulator]:
    """Collect first-pass layer statistics required for norms/cosine/tau.

    This pass computes robust layer-level scale estimates without storing task
    vectors in memory.

    Args:
        base_model: Base checkpoint model.
        if_model: IF-tuned checkpoint model.
        math_model: Math-tuned checkpoint model.

    Returns:
        Dictionary mapping layer name to `Pass1Accumulator`.
    """

    if_params = dict(if_model.named_parameters())
    math_params = dict(math_model.named_parameters())

    accumulators: Dict[str, Pass1Accumulator] = defaultdict(Pass1Accumulator)

    with torch.no_grad():
        for name, base_param in tqdm(base_model.named_parameters(), desc="Pass-1 stats"):
            if name not in if_params or name not in math_params:
                continue
            if if_params[name].shape != base_param.shape or math_params[name].shape != base_param.shape:
                raise ValueError(f"Shape mismatch at parameter: {name}")
            if not should_include_parameter_in_analysis(name, base_param):
                continue

            layer_name = parse_layer_name(name)
            acc = accumulators[layer_name]

            base_fp32 = base_param.detach().to(torch.float32)
            delta_if = if_params[name].detach().to(torch.float32) - base_fp32
            delta_math = math_params[name].detach().to(torch.float32) - base_fp32

            acc.sum_sq_if += torch.sum(delta_if * delta_if).item()
            acc.sum_sq_math += torch.sum(delta_math * delta_math).item()
            acc.dot_sum += torch.sum(delta_if * delta_math).item()
            acc.numel += delta_if.numel()

    return accumulators


def build_tau_map(pass1: Dict[str, Pass1Accumulator], tau_rms_factor: float) -> Dict[str, float]:
    """Build layer-wise threshold map `tau_l` from pass-1 RMS scale.

    Args:
        pass1: First-pass layer accumulators.
        tau_rms_factor: Multiplier applied to average RMS scale.

    Returns:
        Dictionary layer -> tau_l.
    """

    tau_map: Dict[str, float] = {}
    for layer, acc in pass1.items():
        if acc.numel == 0:
            tau_map[layer] = 0.0
            continue
        rms_if = math.sqrt(max(acc.sum_sq_if, 0.0) / acc.numel)
        rms_math = math.sqrt(max(acc.sum_sq_math, 0.0) / acc.numel)
        tau_map[layer] = tau_rms_factor * 0.5 * (rms_if + rms_math)
    return tau_map


In [ ]:
def collect_conflict_stats(
    base_model: AutoModelForCausalLM,
    if_model: AutoModelForCausalLM,
    math_model: AutoModelForCausalLM,
    tau_map: Dict[str, float],
) -> Dict[str, Dict[str, float]]:
    """Collect layer-wise conflict and non-zero overlap statistics.

    Args:
        base_model: Base checkpoint model.
        if_model: IF checkpoint model.
        math_model: Math checkpoint model.
        tau_map: Layer-specific threshold map.

    Returns:
        Nested dictionary containing conflict counters per layer, including
        non-zero overlap counts used for element-wise overlap analysis.
    """

    if_params = dict(if_model.named_parameters())
    math_params = dict(math_model.named_parameters())

    counters = defaultdict(
        lambda: {
            "conflict_tau_count": 0.0,
            "active_tau_count": 0.0,
            "numel": 0.0,
            "sign_conflict_all_nonzero_count": 0.0,
            "both_nonzero_count": 0.0,
            "if_nonzero_count": 0.0,
            "math_nonzero_count": 0.0,
            "either_nonzero_count": 0.0,
        }
    )

    with torch.no_grad():
        for name, base_param in tqdm(base_model.named_parameters(), desc="Pass-2 conflict"):
            if name not in if_params or name not in math_params:
                continue
            if not should_include_parameter_in_analysis(name, base_param):
                continue

            layer_name = parse_layer_name(name)
            tau_l = float(tau_map.get(layer_name, 0.0))
            layer_counter = counters[layer_name]

            base_fp32 = base_param.detach().to(torch.float32)
            delta_if = if_params[name].detach().to(torch.float32) - base_fp32
            delta_math = math_params[name].detach().to(torch.float32) - base_fp32

            abs_if = torch.abs(delta_if)
            abs_math = torch.abs(delta_math)
            sign_diff = (torch.sign(delta_if) * torch.sign(delta_math)) < 0

            # Formula target: mean[|Δ_if|>τ, |Δ_math|>τ, sign mismatch]
            active_tau = (abs_if > tau_l) & (abs_math > tau_l)
            conflict_tau = active_tau & sign_diff

            # Non-zero support diagnostics quantify how much two updates touch
            # the same coordinates before comparing signs.
            if_nonzero = delta_if != 0
            math_nonzero = delta_math != 0
            both_nonzero = if_nonzero & math_nonzero
            either_nonzero = if_nonzero | math_nonzero
            sign_conflict_all_nonzero = both_nonzero & sign_diff

            layer_counter["conflict_tau_count"] += conflict_tau.sum().item()
            layer_counter["active_tau_count"] += active_tau.sum().item()
            layer_counter["sign_conflict_all_nonzero_count"] += sign_conflict_all_nonzero.sum().item()
            layer_counter["both_nonzero_count"] += both_nonzero.sum().item()
            layer_counter["if_nonzero_count"] += if_nonzero.sum().item()
            layer_counter["math_nonzero_count"] += math_nonzero.sum().item()
            layer_counter["either_nonzero_count"] += either_nonzero.sum().item()
            layer_counter["numel"] += delta_if.numel()

    return counters


def build_layer_dataframe(
    pass1: Dict[str, Pass1Accumulator],
    conflict: Dict[str, Dict[str, float]],
    tau_map: Dict[str, float],
) -> pd.DataFrame:
    """Combine pass-1/pass-2 outputs into one dataframe with overlap diagnostics."""

    rows = []
    for layer in sorted(pass1.keys(), key=layer_sort_key):
        acc = pass1[layer]
        conf = conflict.get(layer, {})

        norm_if = math.sqrt(max(acc.sum_sq_if, 0.0))
        norm_math = math.sqrt(max(acc.sum_sq_math, 0.0))

        # Cosine is undefined when either side is zero. We keep cos=0 for stable downstream
        # ranking, and expose `cos_l_defined` so interpretation remains explicit.
        cos_defined = (norm_if > 0.0) and (norm_math > 0.0)
        if cos_defined:
            denom = max(norm_if * norm_math, 1e-12)
            cosine = acc.dot_sum / denom
            cosine = max(min(cosine, 1.0), -1.0)
        else:
            cosine = 0.0

        combined_sq = acc.sum_sq_if + acc.sum_sq_math + (2.0 * acc.dot_sum)
        norm_combined = math.sqrt(max(combined_sq, 0.0))
        cancellation_ratio = norm_combined / max(norm_if + norm_math, 1e-12)

        numel = max(float(acc.numel), 1.0)
        active_tau_count = float(conf.get("active_tau_count", 0.0))
        conflict_tau_count = float(conf.get("conflict_tau_count", 0.0))
        both_nonzero_count = float(conf.get("both_nonzero_count", 0.0))
        if_nonzero_count = float(conf.get("if_nonzero_count", 0.0))
        math_nonzero_count = float(conf.get("math_nonzero_count", 0.0))
        either_nonzero_count = float(conf.get("either_nonzero_count", 0.0))
        sign_conflict_all_nonzero_count = float(conf.get("sign_conflict_all_nonzero_count", 0.0))

        rows.append(
            {
                "layer": layer,
                "norm_if_l": norm_if,
                "norm_math_l": norm_math,
                "rms_if_l": norm_if / math.sqrt(numel),
                "rms_math_l": norm_math / math.sqrt(numel),
                "cos_l": cosine,
                "cos_l_defined": bool(cos_defined),
                "tau_l": float(tau_map.get(layer, 0.0)),
                "num_parameters": int(numel),
                "active_ratio_tau": active_tau_count / numel,
                "conflict_l_tau_mean": conflict_tau_count / numel,
                "conflict_ratio_among_active_tau": conflict_tau_count / max(active_tau_count, 1.0),
                # Two overlap views are reported intentionally:
                # 1) overlap density over all parameters,
                # 2) Jaccard overlap on non-zero support only.
                "if_nonzero_ratio": if_nonzero_count / numel,
                "math_nonzero_ratio": math_nonzero_count / numel,
                "nonzero_overlap_ratio": both_nonzero_count / numel,
                "nonzero_overlap_jaccard": both_nonzero_count / max(either_nonzero_count, 1.0),
                "sign_conflict_ratio_nonzero": sign_conflict_all_nonzero_count / max(both_nonzero_count, 1.0),
                "sign_conflict_ratio_all": sign_conflict_all_nonzero_count / numel,
                "cancellation_ratio": cancellation_ratio,
            }
        )

    df = pd.DataFrame(rows)
    if not df.empty:
        denom = (df["norm_if_l"] + df["norm_math_l"]).replace(0.0, 1e-12)
        df["norm_if_share"] = df["norm_if_l"] / denom
        df["norm_math_share"] = df["norm_math_l"] / denom
    return df


def compute_lm_head_state_dict_stats(
    base_model: AutoModelForCausalLM,
    if_model: AutoModelForCausalLM,
    math_model: AutoModelForCausalLM,
    tau_rms_factor: float,
) -> Tuple[Dict[str, float] | None, Dict[str, float]]:
    """Compute layer-style stats for `lm_head.weight` directly from state_dict.

    Why this function exists:
    - In Qwen-family checkpoints, `lm_head.weight` can be tied to embeddings and
      omitted from `named_parameters()`.
    - The analysis loop above uses `named_parameters()`, which can silently skip
      lm_head in tied-weight setups.
    - This helper ensures we can still inspect lm_head deltas explicitly.

    Args:
        base_model: Base model checkpoint.
        if_model: IF-tuned checkpoint.
        math_model: Math-tuned checkpoint.
        tau_rms_factor: Threshold scale factor used in conflict stats.

    Returns:
        A tuple `(lm_head_row, debug_payload)`:
        - `lm_head_row`: Row compatible with `layer_df`, or `None` if lm_head key missing.
        - `debug_payload`: Additional diagnostics (e.g., tie checks).
    """

    base_sd = base_model.state_dict()
    if_sd = if_model.state_dict()
    math_sd = math_model.state_dict()

    debug_payload = {
        "lm_head_present": float("lm_head.weight" in base_sd and "lm_head.weight" in if_sd and "lm_head.weight" in math_sd),
        "lm_head_tied_with_embed_base": 0.0,
        "lm_head_tied_with_embed_if": 0.0,
        "lm_head_tied_with_embed_math": 0.0,
        "max_abs_delta_lm_minus_embed_if": None,
        "max_abs_delta_lm_minus_embed_math": None,
    }

    if not ("lm_head.weight" in base_sd and "lm_head.weight" in if_sd and "lm_head.weight" in math_sd):
        return None, debug_payload

    with torch.no_grad():
        base_lm = base_sd["lm_head.weight"].detach().to(torch.float32)
        if_lm = if_sd["lm_head.weight"].detach().to(torch.float32)
        math_lm = math_sd["lm_head.weight"].detach().to(torch.float32)

        delta_if = if_lm - base_lm
        delta_math = math_lm - base_lm

        numel = delta_if.numel()
        norm_if = torch.linalg.norm(delta_if).item()
        norm_math = torch.linalg.norm(delta_math).item()
        rms_if = norm_if / max(math.sqrt(numel), 1e-12)
        rms_math = norm_math / max(math.sqrt(numel), 1e-12)
        tau_l = tau_rms_factor * 0.5 * (rms_if + rms_math)

        cos_defined = (norm_if > 0.0) and (norm_math > 0.0)
        if cos_defined:
            cos_l = torch.sum(delta_if * delta_math).item() / max(norm_if * norm_math, 1e-12)
            cos_l = float(max(min(cos_l, 1.0), -1.0))
        else:
            cos_l = 0.0

        abs_if = torch.abs(delta_if)
        abs_math = torch.abs(delta_math)
        sign_diff = (torch.sign(delta_if) * torch.sign(delta_math)) < 0
        active_tau = (abs_if > tau_l) & (abs_math > tau_l)
        conflict_tau = active_tau & sign_diff

        if_nonzero = delta_if != 0
        math_nonzero = delta_math != 0
        both_nonzero = if_nonzero & math_nonzero
        either_nonzero = if_nonzero | math_nonzero
        sign_conflict_all_nonzero = both_nonzero & sign_diff

        combined_sq = torch.sum((delta_if + delta_math) ** 2).item()
        cancellation_ratio = math.sqrt(max(combined_sq, 0.0)) / max(norm_if + norm_math, 1e-12)

        lm_head_row = {
            "layer": "layer_lm_head",
            "norm_if_l": float(norm_if),
            "norm_math_l": float(norm_math),
            "rms_if_l": float(rms_if),
            "rms_math_l": float(rms_math),
            "cos_l": float(cos_l),
            "cos_l_defined": bool(cos_defined),
            "tau_l": float(tau_l),
            "num_parameters": int(numel),
            "active_ratio_tau": float(active_tau.float().mean().item()),
            "conflict_l_tau_mean": float(conflict_tau.float().mean().item()),
            "conflict_ratio_among_active_tau": float(conflict_tau.sum().item() / max(active_tau.sum().item(), 1.0)),
            "if_nonzero_ratio": float(if_nonzero.float().mean().item()),
            "math_nonzero_ratio": float(math_nonzero.float().mean().item()),
            "nonzero_overlap_ratio": float(both_nonzero.float().mean().item()),
            "nonzero_overlap_jaccard": float(both_nonzero.sum().item() / max(either_nonzero.sum().item(), 1.0)),
            "sign_conflict_ratio_nonzero": float(
                sign_conflict_all_nonzero.sum().item() / max(both_nonzero.sum().item(), 1.0)
            ),
            "sign_conflict_ratio_all": float(sign_conflict_all_nonzero.float().mean().item()),
            "cancellation_ratio": float(cancellation_ratio),
        }

        # Tie diagnostics versus embeddings (if present in all checkpoints).
        if (
            "model.embed_tokens.weight" in base_sd
            and "model.embed_tokens.weight" in if_sd
            and "model.embed_tokens.weight" in math_sd
        ):
            base_emb = base_sd["model.embed_tokens.weight"].detach().to(torch.float32)
            if_emb = if_sd["model.embed_tokens.weight"].detach().to(torch.float32)
            math_emb = math_sd["model.embed_tokens.weight"].detach().to(torch.float32)

            debug_payload["lm_head_tied_with_embed_base"] = float(torch.equal(base_lm, base_emb))
            debug_payload["lm_head_tied_with_embed_if"] = float(torch.equal(if_lm, if_emb))
            debug_payload["lm_head_tied_with_embed_math"] = float(torch.equal(math_lm, math_emb))
            debug_payload["max_abs_delta_lm_minus_embed_if"] = float(
                torch.max(torch.abs((if_lm - base_lm) - (if_emb - base_emb))).item()
            )
            debug_payload["max_abs_delta_lm_minus_embed_math"] = float(
                torch.max(torch.abs((math_lm - base_lm) - (math_emb - base_emb))).item()
            )

        return lm_head_row, debug_payload


In [ ]:
def visualize_layer_interference(df: pd.DataFrame, output_dir: Path) -> None:
    """Create and save standard layer-wise interference diagnostic plots.

    Args:
        df: Layer statistics dataframe.
        output_dir: Directory where plots are written.
    """

    if df.empty:
        raise ValueError("Input dataframe is empty. Nothing to visualize.")

    plot_df = df.copy()
    plot_df["plot_idx"] = range(len(plot_df))

    fig, axes = plt.subplots(2, 2, figsize=(20, 12), constrained_layout=True)

    # Plot 1: raw task-vector norm profile by layer (log scale recommended).
    axes[0, 0].plot(plot_df["plot_idx"], plot_df["norm_if_l"], marker="o", label="IF ||Δ||")
    axes[0, 0].plot(plot_df["plot_idx"], plot_df["norm_math_l"], marker="o", label="Math ||Δ||")
    axes[0, 0].set_yscale("log")
    axes[0, 0].set_title("Layer-Wise Task Vector Norms (Raw)")
    axes[0, 0].set_xlabel("Layer index")
    axes[0, 0].set_ylabel("L2 norm (log scale)")
    axes[0, 0].legend()

    # Plot 2: normalized dominance for fair comparison across layers.
    axes[0, 1].plot(plot_df["plot_idx"], plot_df["norm_if_share"], marker="o", label="IF share")
    axes[0, 1].plot(plot_df["plot_idx"], plot_df["norm_math_share"], marker="o", label="Math share")
    axes[0, 1].axhline(0.5, linestyle="--", color="gray", linewidth=1)
    axes[0, 1].set_ylim(0.0, 1.0)
    axes[0, 1].set_title("Task Dominance by Layer (Normalized)")
    axes[0, 1].set_xlabel("Layer index")
    axes[0, 1].set_ylabel("Normalized norm share")
    axes[0, 1].legend()

    # Plot 3: directional compatibility.
    axes[1, 0].plot(plot_df["plot_idx"], plot_df["cos_l"], marker="o", label="cos(Δ_if, Δ_math)")
    axes[1, 0].plot(plot_df["plot_idx"], plot_df["cancellation_ratio"], marker="o", label="cancellation ratio")
    axes[1, 0].axhline(0.0, linestyle="--", color="gray", linewidth=1)
    axes[1, 0].set_ylim(-1.05, 1.05)
    axes[1, 0].set_title("Directional Alignment and Cancellation")
    axes[1, 0].set_xlabel("Layer index")
    axes[1, 0].set_ylabel("Score")
    axes[1, 0].legend()

    # Plot 4: conflict + non-zero overlap diagnostics.
    axes[1, 1].plot(plot_df["plot_idx"], plot_df["conflict_l_tau_mean"], marker="o", label="conflict_l(τ)")
    axes[1, 1].plot(
        plot_df["plot_idx"],
        plot_df["conflict_ratio_among_active_tau"],
        marker="o",
        label="conflict ratio | active",
    )
    axes[1, 1].plot(
        plot_df["plot_idx"],
        plot_df["nonzero_overlap_ratio"],
        marker="o",
        label="non-zero overlap / all",
    )
    axes[1, 1].plot(
        plot_df["plot_idx"],
        plot_df["nonzero_overlap_jaccard"],
        marker="o",
        label="non-zero overlap (Jaccard)",
    )
    axes[1, 1].plot(
        plot_df["plot_idx"],
        plot_df["sign_conflict_ratio_nonzero"],
        marker="o",
        label="sign conflict | overlap",
    )
    axes[1, 1].set_ylim(0.0, 1.0)
    axes[1, 1].set_title("Sign Conflict and Non-Zero Overlap")
    axes[1, 1].set_xlabel("Layer index")
    axes[1, 1].set_ylabel("Ratio")
    axes[1, 1].legend()

    fig_path = output_dir / "layer_interference_overview.png"
    fig.savefig(fig_path, dpi=180)
    plt.show()
    print(f"Saved figure: {fig_path}")


In [ ]:
# --------------------------------------------------------------------------------------
# Execute full layer interference analysis
# --------------------------------------------------------------------------------------
base_model = load_causal_lm(BASE_MODEL_ID, dtype=torch.float16)
if_model = load_causal_lm(IF_MODEL_PATH, dtype=torch.float16)
math_model = load_causal_lm(MATH_MODEL_PATH, dtype=torch.float16)

pass1_stats = collect_pass1_stats(base_model=base_model, if_model=if_model, math_model=math_model)
tau_per_layer = build_tau_map(pass1=pass1_stats, tau_rms_factor=TAU_RMS_FACTOR)
conflict_stats = collect_conflict_stats(
    base_model=base_model,
    if_model=if_model,
    math_model=math_model,
    tau_map=tau_per_layer,
)
layer_df = build_layer_dataframe(pass1=pass1_stats, conflict=conflict_stats, tau_map=tau_per_layer)

# Explicitly compute lm_head statistics from state_dict so tied-weight models
# (where lm_head is absent in named_parameters) are still analyzed.
lm_head_row, lm_head_debug = compute_lm_head_state_dict_stats(
    base_model=base_model,
    if_model=if_model,
    math_model=math_model,
    tau_rms_factor=TAU_RMS_FACTOR,
)
if lm_head_row is not None:
    # Drop old lm_head row first (if any), then append state_dict-based row.
    layer_df = layer_df[layer_df["layer"] != "layer_lm_head"].copy()
    layer_df = pd.concat([layer_df, pd.DataFrame([lm_head_row])], ignore_index=True)

# Safety filter: enforce user-requested exclusion of final norm even if parsing logic changes.
layer_df = layer_df[layer_df["layer"] != "layer_final_norm"].copy()
layer_df = layer_df.sort_values("layer", key=lambda col: col.map(layer_sort_key)).reset_index(drop=True)

# Update shares after any row modifications.
denom = (layer_df["norm_if_l"] + layer_df["norm_math_l"]).replace(0.0, 1e-12)
layer_df["norm_if_share"] = layer_df["norm_if_l"] / denom
layer_df["norm_math_share"] = layer_df["norm_math_l"] / denom

layer_df_path = ARTIFACT_DIR / "layer_interference_stats.csv"
layer_df.to_csv(layer_df_path, index=False)
print(f"Saved stats: {layer_df_path}")
print(f"Contains layer_lm_head: {'layer_lm_head' in set(layer_df['layer'].tolist())}")
print(f"Contains layer_final_norm: {'layer_final_norm' in set(layer_df['layer'].tolist())}")
print("lm_head debug:", lm_head_debug)

# Save explicit lm_head debug artifact for reproducibility.
with (ARTIFACT_DIR / "lm_head_delta_debug.json").open("w", encoding="utf-8") as f:
    json.dump(lm_head_debug, f, indent=2)

display(layer_df)
visualize_layer_interference(df=layer_df, output_dir=ARTIFACT_DIR)

# Explicit cleanup for long-running notebook kernels.
del base_model
if "if_model" in locals():
    del if_model
if "math_model" in locals():
    del math_model

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
# Surface the highest-risk layers based on conflict and cancellation.
report_df = layer_df.copy()
report_df["risk_score"] = (
    report_df["conflict_l_tau_mean"].rank(pct=True)
    + report_df["sign_conflict_ratio_nonzero"].rank(pct=True)
    + (1.0 - report_df["cancellation_ratio"]).rank(pct=True)
    + (1.0 - (report_df["cos_l"] + 1.0) * 0.5).rank(pct=True)
)

risk_report_path = ARTIFACT_DIR / "layer_risk_ranking.csv"
report_df.sort_values("risk_score", ascending=False).to_csv(risk_report_path, index=False)
print(f"Saved risk ranking: {risk_report_path}")
display(report_df.sort_values("risk_score", ascending=False).head(12))


## Continual Stage Vector Analysis (`math-base` vs `math_to_if-math`)

This section applies the same interference diagnostics to sequential continual-learning updates:

1. **Stage-1 update vector**: `Δ_stage1 = θ_math - θ_base`
2. **Stage-2 update vector**: `Δ_stage2 = θ_math_to_if - θ_math`

Goal:
- Compare update energy, alignment, and sign conflict between the two continual stages.
- Localize layers where second-stage IF tuning conflicts with first-stage Math adaptation.


In [ ]:
# --------------------------------------------------------------------------------------
# Continual stage analysis: Stage-1 (math-base) vs Stage-2 (math_to_if-math)
# --------------------------------------------------------------------------------------
MATH_TO_IF_MODEL_PATH = Path(
    "/mnt/nappipe/users/jynam/geeho/nemotron_cascade_output/Qwen3-1.7B-math-if/global_step_50/actor/huggingface"
)
CONTINUAL_ARTIFACT_DIR = Path("merging_analysis/artifacts/layer_interference_continual_stage")
CONTINUAL_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

if not MATH_TO_IF_MODEL_PATH.exists():
    raise FileNotFoundError(f"Math->IF checkpoint not found: {MATH_TO_IF_MODEL_PATH}")


def collect_continual_stage_pass1_stats(
    base_model: AutoModelForCausalLM,
    math_model: AutoModelForCausalLM,
    math_to_if_model: AutoModelForCausalLM,
) -> Dict[str, Pass1Accumulator]:
    """Collect first-pass statistics for continual stage vectors.

    Stage vector definitions:
    - stage-1: `delta_stage1 = theta_math - theta_base`
    - stage-2: `delta_stage2 = theta_math_to_if - theta_math`

    The function reuses `Pass1Accumulator` fields as:
    - `sum_sq_if` -> `sum_sq_stage1`
    - `sum_sq_math` -> `sum_sq_stage2`

    Args:
        base_model: Base model checkpoint.
        math_model: Math-tuned checkpoint.
        math_to_if_model: Continual checkpoint initialized from math then tuned on IF.

    Returns:
        Layer-wise pass1 accumulator dictionary.
    """

    base_params = dict(base_model.named_parameters())
    math_params = dict(math_model.named_parameters())
    math_to_if_params = dict(math_to_if_model.named_parameters())

    accumulators: Dict[str, Pass1Accumulator] = defaultdict(Pass1Accumulator)

    with torch.no_grad():
        for name, base_param in tqdm(base_model.named_parameters(), desc="Stage pass-1 stats"):
            if name not in math_params or name not in math_to_if_params:
                continue
            if not should_include_parameter_in_analysis(name, base_param):
                continue

            math_param = math_params[name]
            math_to_if_param = math_to_if_params[name]
            if math_param.shape != base_param.shape or math_to_if_param.shape != base_param.shape:
                raise ValueError(f"Shape mismatch at parameter: {name}")

            layer_name = parse_layer_name(name)
            acc = accumulators[layer_name]

            base_fp32 = base_param.detach().to(torch.float32)
            math_fp32 = math_param.detach().to(torch.float32)
            math_to_if_fp32 = math_to_if_param.detach().to(torch.float32)

            # Stage-1 update: math - base
            delta_stage1 = math_fp32 - base_fp32
            # Stage-2 update: math_to_if - math
            delta_stage2 = math_to_if_fp32 - math_fp32

            acc.sum_sq_if += torch.sum(delta_stage1 * delta_stage1).item()
            acc.sum_sq_math += torch.sum(delta_stage2 * delta_stage2).item()
            acc.dot_sum += torch.sum(delta_stage1 * delta_stage2).item()
            acc.numel += delta_stage1.numel()

    return accumulators


def build_continual_stage_tau_map(
    pass1: Dict[str, Pass1Accumulator],
    tau_rms_factor: float,
) -> Dict[str, float]:
    """Build layer-wise tau map for continual stage vectors.

    Tau definition per layer `l`:
    - `tau_l = tau_rms_factor * 0.5 * (rms_stage1_l + rms_stage2_l)`

    Args:
        pass1: Pass1 accumulator dictionary.
        tau_rms_factor: Scalar threshold factor.

    Returns:
        Layer -> tau_l map.
    """

    tau_map: Dict[str, float] = {}
    for layer, acc in pass1.items():
        if acc.numel == 0:
            tau_map[layer] = 0.0
            continue
        rms_stage1 = math.sqrt(max(acc.sum_sq_if, 0.0) / acc.numel)
        rms_stage2 = math.sqrt(max(acc.sum_sq_math, 0.0) / acc.numel)
        tau_map[layer] = tau_rms_factor * 0.5 * (rms_stage1 + rms_stage2)
    return tau_map


def collect_continual_stage_conflict_stats(
    base_model: AutoModelForCausalLM,
    math_model: AutoModelForCausalLM,
    math_to_if_model: AutoModelForCausalLM,
    tau_map: Dict[str, float],
) -> Dict[str, Dict[str, float]]:
    """Collect thresholded conflict and non-zero overlap stats for stage vectors.

    Args:
        base_model: Base model checkpoint.
        math_model: Math-tuned checkpoint.
        math_to_if_model: Continual Math->IF checkpoint.
        tau_map: Layer-wise threshold map.

    Returns:
        Layer-wise conflict counters including non-zero overlap statistics.
    """

    math_params = dict(math_model.named_parameters())
    math_to_if_params = dict(math_to_if_model.named_parameters())

    counters = defaultdict(
        lambda: {
            "conflict_tau_count": 0.0,
            "active_tau_count": 0.0,
            "numel": 0.0,
            "sign_conflict_all_nonzero_count": 0.0,
            "both_nonzero_count": 0.0,
            "stage1_nonzero_count": 0.0,
            "stage2_nonzero_count": 0.0,
            "either_nonzero_count": 0.0,
        }
    )

    with torch.no_grad():
        for name, base_param in tqdm(base_model.named_parameters(), desc="Stage pass-2 conflict"):
            if name not in math_params or name not in math_to_if_params:
                continue
            if not should_include_parameter_in_analysis(name, base_param):
                continue

            math_param = math_params[name]
            math_to_if_param = math_to_if_params[name]
            if math_param.shape != base_param.shape or math_to_if_param.shape != base_param.shape:
                raise ValueError(f"Shape mismatch at parameter: {name}")

            layer_name = parse_layer_name(name)
            tau_l = float(tau_map.get(layer_name, 0.0))
            layer_counter = counters[layer_name]

            base_fp32 = base_param.detach().to(torch.float32)
            math_fp32 = math_param.detach().to(torch.float32)
            math_to_if_fp32 = math_to_if_param.detach().to(torch.float32)

            delta_stage1 = math_fp32 - base_fp32
            delta_stage2 = math_to_if_fp32 - math_fp32

            abs_stage1 = torch.abs(delta_stage1)
            abs_stage2 = torch.abs(delta_stage2)
            sign_diff = (torch.sign(delta_stage1) * torch.sign(delta_stage2)) < 0

            active_tau = (abs_stage1 > tau_l) & (abs_stage2 > tau_l)
            conflict_tau = active_tau & sign_diff

            stage1_nonzero = delta_stage1 != 0
            stage2_nonzero = delta_stage2 != 0
            both_nonzero = stage1_nonzero & stage2_nonzero
            either_nonzero = stage1_nonzero | stage2_nonzero
            sign_conflict_nonzero = both_nonzero & sign_diff

            layer_counter["conflict_tau_count"] += conflict_tau.sum().item()
            layer_counter["active_tau_count"] += active_tau.sum().item()
            layer_counter["sign_conflict_all_nonzero_count"] += sign_conflict_nonzero.sum().item()
            layer_counter["both_nonzero_count"] += both_nonzero.sum().item()
            layer_counter["stage1_nonzero_count"] += stage1_nonzero.sum().item()
            layer_counter["stage2_nonzero_count"] += stage2_nonzero.sum().item()
            layer_counter["either_nonzero_count"] += either_nonzero.sum().item()
            layer_counter["numel"] += delta_stage1.numel()

    return counters


def build_continual_stage_dataframe(
    pass1: Dict[str, Pass1Accumulator],
    conflict: Dict[str, Dict[str, float]],
    tau_map: Dict[str, float],
) -> pd.DataFrame:
    """Create tidy dataframe for continual stage vector diagnostics.

    Output columns mirror the original analysis, but with stage-specific names.

    Args:
        pass1: Stage pass1 accumulator dictionary.
        conflict: Stage conflict counter dictionary.
        tau_map: Stage tau dictionary.

    Returns:
        DataFrame with layer-wise continual stage diagnostics.
    """

    rows = []
    for layer in sorted(pass1.keys(), key=layer_sort_key):
        acc = pass1[layer]
        conf = conflict.get(layer, {})

        norm_stage1 = math.sqrt(max(acc.sum_sq_if, 0.0))
        norm_stage2 = math.sqrt(max(acc.sum_sq_math, 0.0))

        cos_defined = (norm_stage1 > 0.0) and (norm_stage2 > 0.0)
        if cos_defined:
            cos_l = acc.dot_sum / max(norm_stage1 * norm_stage2, 1e-12)
            cos_l = max(min(cos_l, 1.0), -1.0)
        else:
            cos_l = 0.0

        combined_sq = acc.sum_sq_if + acc.sum_sq_math + (2.0 * acc.dot_sum)
        cancellation_ratio = math.sqrt(max(combined_sq, 0.0)) / max(norm_stage1 + norm_stage2, 1e-12)

        numel = max(float(acc.numel), 1.0)
        active_tau_count = float(conf.get("active_tau_count", 0.0))
        conflict_tau_count = float(conf.get("conflict_tau_count", 0.0))
        both_nonzero_count = float(conf.get("both_nonzero_count", 0.0))
        stage1_nonzero_count = float(conf.get("stage1_nonzero_count", 0.0))
        stage2_nonzero_count = float(conf.get("stage2_nonzero_count", 0.0))
        either_nonzero_count = float(conf.get("either_nonzero_count", 0.0))
        sign_conflict_nonzero_count = float(conf.get("sign_conflict_all_nonzero_count", 0.0))

        rows.append(
            {
                "layer": layer,
                "norm_stage1_l": norm_stage1,
                "norm_stage2_l": norm_stage2,
                "rms_stage1_l": norm_stage1 / math.sqrt(numel),
                "rms_stage2_l": norm_stage2 / math.sqrt(numel),
                "cos_l": float(cos_l),
                "cos_l_defined": bool(cos_defined),
                "tau_l": float(tau_map.get(layer, 0.0)),
                "num_parameters": int(numel),
                "active_ratio_tau": active_tau_count / numel,
                "conflict_l_tau_mean": conflict_tau_count / numel,
                "conflict_ratio_among_active_tau": conflict_tau_count / max(active_tau_count, 1.0),
                "stage1_nonzero_ratio": stage1_nonzero_count / numel,
                "stage2_nonzero_ratio": stage2_nonzero_count / numel,
                "nonzero_overlap_ratio": both_nonzero_count / numel,
                "nonzero_overlap_jaccard": both_nonzero_count / max(either_nonzero_count, 1.0),
                "sign_conflict_ratio_nonzero": sign_conflict_nonzero_count / max(both_nonzero_count, 1.0),
                "sign_conflict_ratio_all": sign_conflict_nonzero_count / numel,
                "cancellation_ratio": cancellation_ratio,
            }
        )

    df = pd.DataFrame(rows)
    if not df.empty:
        denom = (df["norm_stage1_l"] + df["norm_stage2_l"]).replace(0.0, 1e-12)
        df["norm_stage1_share"] = df["norm_stage1_l"] / denom
        df["norm_stage2_share"] = df["norm_stage2_l"] / denom

    return df


def compute_continual_stage_lm_head_stats(
    base_model: AutoModelForCausalLM,
    math_model: AutoModelForCausalLM,
    math_to_if_model: AutoModelForCausalLM,
    tau_rms_factor: float,
) -> Tuple[Dict[str, float] | None, Dict[str, float]]:
    """Compute `lm_head` stage-vector stats from state_dict keys directly.

    This is required because tied-weight models can omit `lm_head.weight` from
    `named_parameters()`, even though the key exists in `state_dict()`.

    Args:
        base_model: Base model checkpoint.
        math_model: Math-tuned checkpoint.
        math_to_if_model: Continual Math->IF checkpoint.
        tau_rms_factor: Scalar threshold factor.

    Returns:
        Tuple `(lm_head_row, debug_payload)`.
    """

    base_sd = base_model.state_dict()
    math_sd = math_model.state_dict()
    math_to_if_sd = math_to_if_model.state_dict()

    debug_payload = {
        "lm_head_present": float(
            "lm_head.weight" in base_sd and "lm_head.weight" in math_sd and "lm_head.weight" in math_to_if_sd
        ),
        "lm_head_tied_with_embed_base": 0.0,
        "lm_head_tied_with_embed_math": 0.0,
        "lm_head_tied_with_embed_math_to_if": 0.0,
        "max_abs_stage1_delta_lm_minus_embed": None,
        "max_abs_stage2_delta_lm_minus_embed": None,
    }

    if debug_payload["lm_head_present"] == 0.0:
        return None, debug_payload

    with torch.no_grad():
        base_lm = base_sd["lm_head.weight"].detach().to(torch.float32)
        math_lm = math_sd["lm_head.weight"].detach().to(torch.float32)
        math_to_if_lm = math_to_if_sd["lm_head.weight"].detach().to(torch.float32)

        delta_stage1 = math_lm - base_lm
        delta_stage2 = math_to_if_lm - math_lm

        numel = delta_stage1.numel()
        norm_stage1 = torch.linalg.norm(delta_stage1).item()
        norm_stage2 = torch.linalg.norm(delta_stage2).item()
        rms_stage1 = norm_stage1 / max(math.sqrt(numel), 1e-12)
        rms_stage2 = norm_stage2 / max(math.sqrt(numel), 1e-12)
        tau_l = tau_rms_factor * 0.5 * (rms_stage1 + rms_stage2)

        cos_defined = (norm_stage1 > 0.0) and (norm_stage2 > 0.0)
        if cos_defined:
            cos_l = torch.sum(delta_stage1 * delta_stage2).item() / max(norm_stage1 * norm_stage2, 1e-12)
            cos_l = max(min(cos_l, 1.0), -1.0)
        else:
            cos_l = 0.0

        sign_diff = (torch.sign(delta_stage1) * torch.sign(delta_stage2)) < 0
        active_tau = (torch.abs(delta_stage1) > tau_l) & (torch.abs(delta_stage2) > tau_l)
        conflict_tau = active_tau & sign_diff
        stage1_nonzero = delta_stage1 != 0
        stage2_nonzero = delta_stage2 != 0
        both_nonzero = stage1_nonzero & stage2_nonzero
        either_nonzero = stage1_nonzero | stage2_nonzero
        sign_conflict_nonzero = both_nonzero & sign_diff

        combined_sq = torch.sum((delta_stage1 + delta_stage2) ** 2).item()
        cancellation_ratio = math.sqrt(max(combined_sq, 0.0)) / max(norm_stage1 + norm_stage2, 1e-12)

        lm_head_row = {
            "layer": "layer_lm_head",
            "norm_stage1_l": float(norm_stage1),
            "norm_stage2_l": float(norm_stage2),
            "rms_stage1_l": float(rms_stage1),
            "rms_stage2_l": float(rms_stage2),
            "cos_l": float(cos_l),
            "cos_l_defined": bool(cos_defined),
            "tau_l": float(tau_l),
            "num_parameters": int(numel),
            "active_ratio_tau": float(active_tau.float().mean().item()),
            "conflict_l_tau_mean": float(conflict_tau.float().mean().item()),
            "conflict_ratio_among_active_tau": float(conflict_tau.sum().item() / max(active_tau.sum().item(), 1.0)),
            "stage1_nonzero_ratio": float(stage1_nonzero.float().mean().item()),
            "stage2_nonzero_ratio": float(stage2_nonzero.float().mean().item()),
            "nonzero_overlap_ratio": float(both_nonzero.float().mean().item()),
            "nonzero_overlap_jaccard": float(both_nonzero.sum().item() / max(either_nonzero.sum().item(), 1.0)),
            "sign_conflict_ratio_nonzero": float(
                sign_conflict_nonzero.sum().item() / max(both_nonzero.sum().item(), 1.0)
            ),
            "sign_conflict_ratio_all": float(sign_conflict_nonzero.float().mean().item()),
            "cancellation_ratio": float(cancellation_ratio),
        }

        if (
            "model.embed_tokens.weight" in base_sd
            and "model.embed_tokens.weight" in math_sd
            and "model.embed_tokens.weight" in math_to_if_sd
        ):
            base_emb = base_sd["model.embed_tokens.weight"].detach().to(torch.float32)
            math_emb = math_sd["model.embed_tokens.weight"].detach().to(torch.float32)
            math_to_if_emb = math_to_if_sd["model.embed_tokens.weight"].detach().to(torch.float32)

            debug_payload["lm_head_tied_with_embed_base"] = float(torch.equal(base_lm, base_emb))
            debug_payload["lm_head_tied_with_embed_math"] = float(torch.equal(math_lm, math_emb))
            debug_payload["lm_head_tied_with_embed_math_to_if"] = float(torch.equal(math_to_if_lm, math_to_if_emb))
            debug_payload["max_abs_stage1_delta_lm_minus_embed"] = float(
                torch.max(torch.abs((math_lm - base_lm) - (math_emb - base_emb))).item()
            )
            debug_payload["max_abs_stage2_delta_lm_minus_embed"] = float(
                torch.max(torch.abs((math_to_if_lm - math_lm) - (math_to_if_emb - math_emb))).item()
            )

        return lm_head_row, debug_payload


def visualize_continual_stage_interference(df: pd.DataFrame, output_dir: Path) -> None:
    """Visualize continual stage vector interference diagnostics.

    Args:
        df: Continual-stage diagnostics dataframe.
        output_dir: Directory where the overview figure is saved.
    """

    if df.empty:
        raise ValueError("Input dataframe is empty. Nothing to visualize.")

    plot_df = df.copy()
    plot_df["plot_idx"] = range(len(plot_df))

    fig, axes = plt.subplots(2, 2, figsize=(20, 12), constrained_layout=True)

    # Plot 1: raw stage vector norms.
    axes[0, 0].plot(plot_df["plot_idx"], plot_df["norm_stage1_l"], marker="o", label="Stage-1 ||Δ|| (math-base)")
    axes[0, 0].plot(plot_df["plot_idx"], plot_df["norm_stage2_l"], marker="o", label="Stage-2 ||Δ|| (math_to_if-math)")
    axes[0, 0].set_yscale("log")
    axes[0, 0].set_title("Layer-Wise Stage Vector Norms (Raw)")
    axes[0, 0].set_xlabel("Layer index")
    axes[0, 0].set_ylabel("L2 norm (log scale)")
    axes[0, 0].legend()

    # Plot 2: normalized stage dominance.
    axes[0, 1].plot(plot_df["plot_idx"], plot_df["norm_stage1_share"], marker="o", label="Stage-1 share")
    axes[0, 1].plot(plot_df["plot_idx"], plot_df["norm_stage2_share"], marker="o", label="Stage-2 share")
    axes[0, 1].axhline(0.5, linestyle="--", color="gray", linewidth=1)
    axes[0, 1].set_ylim(0.0, 1.0)
    axes[0, 1].set_title("Stage Dominance by Layer (Normalized)")
    axes[0, 1].set_xlabel("Layer index")
    axes[0, 1].set_ylabel("Normalized norm share")
    axes[0, 1].legend()

    # Plot 3: directional compatibility and cancellation.
    axes[1, 0].plot(plot_df["plot_idx"], plot_df["cos_l"], marker="o", label="cos(Δ_stage1, Δ_stage2)")
    axes[1, 0].plot(plot_df["plot_idx"], plot_df["cancellation_ratio"], marker="o", label="cancellation ratio")
    axes[1, 0].axhline(0.0, linestyle="--", color="gray", linewidth=1)
    axes[1, 0].set_ylim(-1.05, 1.05)
    axes[1, 0].set_title("Directional Alignment and Cancellation")
    axes[1, 0].set_xlabel("Layer index")
    axes[1, 0].set_ylabel("Score")
    axes[1, 0].legend()

    # Plot 4: thresholded conflict + non-zero overlap statistics.
    axes[1, 1].plot(plot_df["plot_idx"], plot_df["conflict_l_tau_mean"], marker="o", label="conflict_l(τ)")
    axes[1, 1].plot(
        plot_df["plot_idx"],
        plot_df["conflict_ratio_among_active_tau"],
        marker="o",
        label="conflict ratio | active",
    )
    axes[1, 1].plot(
        plot_df["plot_idx"],
        plot_df["nonzero_overlap_ratio"],
        marker="o",
        label="non-zero overlap / all",
    )
    axes[1, 1].plot(
        plot_df["plot_idx"],
        plot_df["nonzero_overlap_jaccard"],
        marker="o",
        label="non-zero overlap (Jaccard)",
    )
    axes[1, 1].plot(
        plot_df["plot_idx"],
        plot_df["sign_conflict_ratio_nonzero"],
        marker="o",
        label="sign conflict | overlap",
    )
    axes[1, 1].set_ylim(0.0, 1.0)
    axes[1, 1].set_title("Sign Conflict and Non-Zero Overlap")
    axes[1, 1].set_xlabel("Layer index")
    axes[1, 1].set_ylabel("Ratio")
    axes[1, 1].legend()

    figure_path = output_dir / "continual_stage_interference_overview.png"
    fig.savefig(figure_path, dpi=180)
    plt.show()
    print(f"Saved figure: {figure_path}")


# ----------------------------------
# Execute continual stage diagnostics
# ----------------------------------
base_model_stage = load_causal_lm(BASE_MODEL_ID, dtype=torch.float16)
math_model_stage = load_causal_lm(MATH_MODEL_PATH, dtype=torch.float16)
math_to_if_model_stage = load_causal_lm(MATH_TO_IF_MODEL_PATH, dtype=torch.float16)

pass1_stage = collect_continual_stage_pass1_stats(
    base_model=base_model_stage,
    math_model=math_model_stage,
    math_to_if_model=math_to_if_model_stage,
)
tau_stage = build_continual_stage_tau_map(pass1=pass1_stage, tau_rms_factor=TAU_RMS_FACTOR)
conflict_stage = collect_continual_stage_conflict_stats(
    base_model=base_model_stage,
    math_model=math_model_stage,
    math_to_if_model=math_to_if_model_stage,
    tau_map=tau_stage,
)
stage_df = build_continual_stage_dataframe(pass1=pass1_stage, conflict=conflict_stage, tau_map=tau_stage)

# Add lm_head row from state_dict to support tied-weight architectures.
stage_lm_head_row, stage_lm_head_debug = compute_continual_stage_lm_head_stats(
    base_model=base_model_stage,
    math_model=math_model_stage,
    math_to_if_model=math_to_if_model_stage,
    tau_rms_factor=TAU_RMS_FACTOR,
)
if stage_lm_head_row is not None:
    stage_df = stage_df[stage_df["layer"] != "layer_lm_head"].copy()
    stage_df = pd.concat([stage_df, pd.DataFrame([stage_lm_head_row])], ignore_index=True)

# Safety filter: keep final_norm excluded for stable interpretation.
stage_df = stage_df[stage_df["layer"] != "layer_final_norm"].copy()
stage_df = stage_df.sort_values("layer", key=lambda col: col.map(layer_sort_key)).reset_index(drop=True)

# Recompute shares after row adjustments.
denom = (stage_df["norm_stage1_l"] + stage_df["norm_stage2_l"]).replace(0.0, 1e-12)
stage_df["norm_stage1_share"] = stage_df["norm_stage1_l"] / denom
stage_df["norm_stage2_share"] = stage_df["norm_stage2_l"] / denom

# Stage risk ranking (same spirit as original analysis).
stage_report_df = stage_df.copy()
stage_report_df["risk_score"] = (
    stage_report_df["conflict_l_tau_mean"].rank(pct=True)
    + stage_report_df["sign_conflict_ratio_nonzero"].rank(pct=True)
    + (1.0 - stage_report_df["cancellation_ratio"]).rank(pct=True)
    + (1.0 - (stage_report_df["cos_l"] + 1.0) * 0.5).rank(pct=True)
)

stage_csv_path = CONTINUAL_ARTIFACT_DIR / "continual_stage_layer_interference_stats.csv"
stage_risk_csv_path = CONTINUAL_ARTIFACT_DIR / "continual_stage_layer_risk_ranking.csv"
stage_debug_json_path = CONTINUAL_ARTIFACT_DIR / "continual_stage_lm_head_debug.json"

stage_df.to_csv(stage_csv_path, index=False)
stage_report_df.sort_values("risk_score", ascending=False).to_csv(stage_risk_csv_path, index=False)
with stage_debug_json_path.open("w", encoding="utf-8") as f:
    json.dump(stage_lm_head_debug, f, indent=2)

print(f"Saved stage stats: {stage_csv_path}")
print(f"Saved stage risk ranking: {stage_risk_csv_path}")
print(f"Saved stage lm_head debug: {stage_debug_json_path}")
print(f"Contains layer_lm_head: {'layer_lm_head' in set(stage_df['layer'].tolist())}")
print(f"Contains layer_final_norm: {'layer_final_norm' in set(stage_df['layer'].tolist())}")
print("Stage lm_head debug:", stage_lm_head_debug)

display(stage_df)
visualize_continual_stage_interference(df=stage_df, output_dir=CONTINUAL_ARTIFACT_DIR)

# Cleanup stage-specific model objects.
del base_model_stage
if "math_model_stage" in locals():
    del math_model_stage
if "math_to_if_model_stage" in locals():
    del math_to_if_model_stage

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## Pairwise Vector Comparison (`if-base` vs `math_to_if-math`)

This section compares the exact vectors below, as requested:

- `delta_if_minus_base = theta_if - theta_base`
- `delta_math_to_if_minus_math = theta_math_to_if - theta_math`

The same diagnostics are reused so results are directly comparable with previous sections.


In [ ]:
# --------------------------------------------------------------------------------------
# Pairwise vector diagnostics: (if - base) vs (math_to_if - math)
# --------------------------------------------------------------------------------------
PAIRWISE_ARTIFACT_DIR = Path("merging_analysis/artifacts/layer_interference_if_base_vs_math_to_if_minus_math")
PAIRWISE_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# Defensive fallback keeps this cell executable even if the previous stage cell was skipped.
if "MATH_TO_IF_MODEL_PATH" not in globals():
    MATH_TO_IF_MODEL_PATH = Path(
        "/mnt/nappipe/users/jynam/geeho/nemotron_cascade_output/Qwen3-1.7B-math-if/global_step_50/actor/huggingface"
    )

if not IF_MODEL_PATH.exists():
    raise FileNotFoundError(f"IF checkpoint not found: {IF_MODEL_PATH}")
if not MATH_MODEL_PATH.exists():
    raise FileNotFoundError(f"Math checkpoint not found: {MATH_MODEL_PATH}")
if not MATH_TO_IF_MODEL_PATH.exists():
    raise FileNotFoundError(f"Math->IF checkpoint not found: {MATH_TO_IF_MODEL_PATH}")


def collect_ifbase_vs_stage2_pass1_stats(
    base_model: AutoModelForCausalLM,
    if_model: AutoModelForCausalLM,
    math_model: AutoModelForCausalLM,
    math_to_if_model: AutoModelForCausalLM,
) -> Dict[str, Pass1Accumulator]:
    """Collect pass-1 stats for two vectors with different anchors.

    Vector definitions:
    - vector A (stored in `sum_sq_if`): `delta_if_minus_base = theta_if - theta_base`
    - vector B (stored in `sum_sq_math`): `delta_math_to_if_minus_math = theta_math_to_if - theta_math`

    We keep the accumulator field names (`if`/`math`) for compatibility with
    existing dataframe/visualization helpers, then add alias columns later for clarity.

    Args:
        base_model: Base checkpoint.
        if_model: IF checkpoint.
        math_model: Math checkpoint (anchor for stage-2 vector).
        math_to_if_model: Continual math->if checkpoint.

    Returns:
        Layer-wise pass-1 accumulators with norms/dot products and parameter counts.
    """

    if_params = dict(if_model.named_parameters())
    math_params = dict(math_model.named_parameters())
    math_to_if_params = dict(math_to_if_model.named_parameters())

    accumulators: Dict[str, Pass1Accumulator] = defaultdict(Pass1Accumulator)

    with torch.no_grad():
        for name, base_param in tqdm(base_model.named_parameters(), desc="Pairwise pass-1 stats"):
            if name not in if_params or name not in math_params or name not in math_to_if_params:
                continue
            if not should_include_parameter_in_analysis(name, base_param):
                continue

            if_param = if_params[name]
            math_param = math_params[name]
            math_to_if_param = math_to_if_params[name]
            if (
                if_param.shape != base_param.shape
                or math_param.shape != base_param.shape
                or math_to_if_param.shape != base_param.shape
            ):
                raise ValueError(f"Shape mismatch at parameter: {name}")

            base_fp32 = base_param.detach().to(torch.float32)
            if_fp32 = if_param.detach().to(torch.float32)
            math_fp32 = math_param.detach().to(torch.float32)
            math_to_if_fp32 = math_to_if_param.detach().to(torch.float32)

            delta_if_minus_base = if_fp32 - base_fp32
            delta_math_to_if_minus_math = math_to_if_fp32 - math_fp32

            layer_name = parse_layer_name(name)
            acc = accumulators[layer_name]

            # Reuse `if/math` fields as vector-A/vector-B storage for compatibility.
            acc.sum_sq_if += torch.sum(delta_if_minus_base * delta_if_minus_base).item()
            acc.sum_sq_math += torch.sum(delta_math_to_if_minus_math * delta_math_to_if_minus_math).item()
            acc.dot_sum += torch.sum(delta_if_minus_base * delta_math_to_if_minus_math).item()
            acc.numel += delta_if_minus_base.numel()

    return accumulators


def collect_ifbase_vs_stage2_conflict_stats(
    base_model: AutoModelForCausalLM,
    if_model: AutoModelForCausalLM,
    math_model: AutoModelForCausalLM,
    math_to_if_model: AutoModelForCausalLM,
    tau_map: Dict[str, float],
) -> Dict[str, Dict[str, float]]:
    """Collect sign-conflict and non-zero overlap counters for pairwise vectors.

    Args:
        base_model: Base checkpoint.
        if_model: IF checkpoint.
        math_model: Math checkpoint.
        math_to_if_model: Continual math->if checkpoint.
        tau_map: Layer-wise tau threshold map.

    Returns:
        Layer-wise conflict counters compatible with `build_layer_dataframe`.
    """

    if_params = dict(if_model.named_parameters())
    math_params = dict(math_model.named_parameters())
    math_to_if_params = dict(math_to_if_model.named_parameters())

    counters = defaultdict(
        lambda: {
            "conflict_tau_count": 0.0,
            "active_tau_count": 0.0,
            "numel": 0.0,
            "sign_conflict_all_nonzero_count": 0.0,
            "both_nonzero_count": 0.0,
            "if_nonzero_count": 0.0,
            "math_nonzero_count": 0.0,
            "either_nonzero_count": 0.0,
        }
    )

    with torch.no_grad():
        for name, base_param in tqdm(base_model.named_parameters(), desc="Pairwise pass-2 conflict"):
            if name not in if_params or name not in math_params or name not in math_to_if_params:
                continue
            if not should_include_parameter_in_analysis(name, base_param):
                continue

            if_param = if_params[name]
            math_param = math_params[name]
            math_to_if_param = math_to_if_params[name]
            if (
                if_param.shape != base_param.shape
                or math_param.shape != base_param.shape
                or math_to_if_param.shape != base_param.shape
            ):
                raise ValueError(f"Shape mismatch at parameter: {name}")

            layer_name = parse_layer_name(name)
            tau_l = float(tau_map.get(layer_name, 0.0))
            layer_counter = counters[layer_name]

            base_fp32 = base_param.detach().to(torch.float32)
            if_fp32 = if_param.detach().to(torch.float32)
            math_fp32 = math_param.detach().to(torch.float32)
            math_to_if_fp32 = math_to_if_param.detach().to(torch.float32)

            delta_if_minus_base = if_fp32 - base_fp32
            delta_math_to_if_minus_math = math_to_if_fp32 - math_fp32

            abs_if_base = torch.abs(delta_if_minus_base)
            abs_stage2 = torch.abs(delta_math_to_if_minus_math)
            sign_diff = (torch.sign(delta_if_minus_base) * torch.sign(delta_math_to_if_minus_math)) < 0

            active_tau = (abs_if_base > tau_l) & (abs_stage2 > tau_l)
            conflict_tau = active_tau & sign_diff

            if_nonzero = delta_if_minus_base != 0
            stage2_nonzero = delta_math_to_if_minus_math != 0
            both_nonzero = if_nonzero & stage2_nonzero
            either_nonzero = if_nonzero | stage2_nonzero
            sign_conflict_nonzero = both_nonzero & sign_diff

            layer_counter["conflict_tau_count"] += conflict_tau.sum().item()
            layer_counter["active_tau_count"] += active_tau.sum().item()
            layer_counter["sign_conflict_all_nonzero_count"] += sign_conflict_nonzero.sum().item()
            layer_counter["both_nonzero_count"] += both_nonzero.sum().item()
            layer_counter["if_nonzero_count"] += if_nonzero.sum().item()
            layer_counter["math_nonzero_count"] += stage2_nonzero.sum().item()
            layer_counter["either_nonzero_count"] += either_nonzero.sum().item()
            layer_counter["numel"] += delta_if_minus_base.numel()

    return counters


def compute_ifbase_vs_stage2_lm_head_stats(
    base_model: AutoModelForCausalLM,
    if_model: AutoModelForCausalLM,
    math_model: AutoModelForCausalLM,
    math_to_if_model: AutoModelForCausalLM,
    tau_rms_factor: float,
) -> Tuple[Dict[str, float] | None, Dict[str, float]]:
    """Compute `lm_head` row for `if-base` vs `math_to_if-math` vectors.

    This mirrors the layer-wise formulas, but works on `state_dict()` keys so tied
    output heads are still diagnosable even when omitted from `named_parameters()`.

    Args:
        base_model: Base checkpoint.
        if_model: IF checkpoint.
        math_model: Math checkpoint.
        math_to_if_model: Continual math->if checkpoint.
        tau_rms_factor: Scalar factor for tau threshold construction.

    Returns:
        Tuple `(lm_head_row, debug_payload)`.
    """

    base_sd = base_model.state_dict()
    if_sd = if_model.state_dict()
    math_sd = math_model.state_dict()
    math_to_if_sd = math_to_if_model.state_dict()

    debug_payload = {
        "lm_head_present": float(
            "lm_head.weight" in base_sd
            and "lm_head.weight" in if_sd
            and "lm_head.weight" in math_sd
            and "lm_head.weight" in math_to_if_sd
        ),
        "lm_head_tied_with_embed_base": 0.0,
        "lm_head_tied_with_embed_if": 0.0,
        "lm_head_tied_with_embed_math": 0.0,
        "lm_head_tied_with_embed_math_to_if": 0.0,
        "max_abs_delta_lm_minus_embed_if_minus_base": None,
        "max_abs_delta_lm_minus_embed_math_to_if_minus_math": None,
    }

    if not debug_payload["lm_head_present"]:
        return None, debug_payload

    with torch.no_grad():
        base_lm = base_sd["lm_head.weight"].detach().to(torch.float32)
        if_lm = if_sd["lm_head.weight"].detach().to(torch.float32)
        math_lm = math_sd["lm_head.weight"].detach().to(torch.float32)
        math_to_if_lm = math_to_if_sd["lm_head.weight"].detach().to(torch.float32)

        delta_if_minus_base = if_lm - base_lm
        delta_math_to_if_minus_math = math_to_if_lm - math_lm

        numel = delta_if_minus_base.numel()
        norm_if = torch.linalg.norm(delta_if_minus_base).item()
        norm_math = torch.linalg.norm(delta_math_to_if_minus_math).item()
        rms_if = norm_if / max(math.sqrt(numel), 1e-12)
        rms_math = norm_math / max(math.sqrt(numel), 1e-12)
        tau_l = tau_rms_factor * 0.5 * (rms_if + rms_math)

        cos_defined = (norm_if > 0.0) and (norm_math > 0.0)
        if cos_defined:
            cos_l = torch.sum(delta_if_minus_base * delta_math_to_if_minus_math).item() / max(norm_if * norm_math, 1e-12)
            cos_l = float(max(min(cos_l, 1.0), -1.0))
        else:
            cos_l = 0.0

        abs_if = torch.abs(delta_if_minus_base)
        abs_math = torch.abs(delta_math_to_if_minus_math)
        sign_diff = (torch.sign(delta_if_minus_base) * torch.sign(delta_math_to_if_minus_math)) < 0

        active_tau = (abs_if > tau_l) & (abs_math > tau_l)
        conflict_tau = active_tau & sign_diff

        if_nonzero = delta_if_minus_base != 0
        stage2_nonzero = delta_math_to_if_minus_math != 0
        both_nonzero = if_nonzero & stage2_nonzero
        either_nonzero = if_nonzero | stage2_nonzero
        sign_conflict_all_nonzero = both_nonzero & sign_diff

        combined_sq = torch.sum((delta_if_minus_base + delta_math_to_if_minus_math) ** 2).item()
        cancellation_ratio = math.sqrt(max(combined_sq, 0.0)) / max(norm_if + norm_math, 1e-12)

        lm_head_row = {
            "layer": "layer_lm_head",
            "norm_if_l": float(norm_if),
            "norm_math_l": float(norm_math),
            "rms_if_l": float(rms_if),
            "rms_math_l": float(rms_math),
            "cos_l": float(cos_l),
            "cos_l_defined": bool(cos_defined),
            "tau_l": float(tau_l),
            "num_parameters": int(numel),
            "active_ratio_tau": float(active_tau.float().mean().item()),
            "conflict_l_tau_mean": float(conflict_tau.float().mean().item()),
            "conflict_ratio_among_active_tau": float(conflict_tau.sum().item() / max(active_tau.sum().item(), 1.0)),
            "if_nonzero_ratio": float(if_nonzero.float().mean().item()),
            "math_nonzero_ratio": float(stage2_nonzero.float().mean().item()),
            "nonzero_overlap_ratio": float(both_nonzero.float().mean().item()),
            "nonzero_overlap_jaccard": float(both_nonzero.sum().item() / max(either_nonzero.sum().item(), 1.0)),
            "sign_conflict_ratio_nonzero": float(
                sign_conflict_all_nonzero.sum().item() / max(both_nonzero.sum().item(), 1.0)
            ),
            "sign_conflict_ratio_all": float(sign_conflict_all_nonzero.float().mean().item()),
            "cancellation_ratio": float(cancellation_ratio),
        }

        if (
            "model.embed_tokens.weight" in base_sd
            and "model.embed_tokens.weight" in if_sd
            and "model.embed_tokens.weight" in math_sd
            and "model.embed_tokens.weight" in math_to_if_sd
        ):
            base_emb = base_sd["model.embed_tokens.weight"].detach().to(torch.float32)
            if_emb = if_sd["model.embed_tokens.weight"].detach().to(torch.float32)
            math_emb = math_sd["model.embed_tokens.weight"].detach().to(torch.float32)
            math_to_if_emb = math_to_if_sd["model.embed_tokens.weight"].detach().to(torch.float32)

            debug_payload["lm_head_tied_with_embed_base"] = float(torch.equal(base_lm, base_emb))
            debug_payload["lm_head_tied_with_embed_if"] = float(torch.equal(if_lm, if_emb))
            debug_payload["lm_head_tied_with_embed_math"] = float(torch.equal(math_lm, math_emb))
            debug_payload["lm_head_tied_with_embed_math_to_if"] = float(torch.equal(math_to_if_lm, math_to_if_emb))
            debug_payload["max_abs_delta_lm_minus_embed_if_minus_base"] = float(
                torch.max(torch.abs((if_lm - base_lm) - (if_emb - base_emb))).item()
            )
            debug_payload["max_abs_delta_lm_minus_embed_math_to_if_minus_math"] = float(
                torch.max(torch.abs((math_to_if_lm - math_lm) - (math_to_if_emb - math_emb))).item()
            )

        return lm_head_row, debug_payload


# Load all checkpoints explicitly because each vector uses a different anchor model.
base_model_pairwise = load_causal_lm(BASE_MODEL_ID, dtype=torch.float16)
if_model_pairwise = load_causal_lm(IF_MODEL_PATH, dtype=torch.float16)
math_model_pairwise = load_causal_lm(MATH_MODEL_PATH, dtype=torch.float16)
math_to_if_pairwise = load_causal_lm(MATH_TO_IF_MODEL_PATH, dtype=torch.float16)

pass1_pairwise = collect_ifbase_vs_stage2_pass1_stats(
    base_model=base_model_pairwise,
    if_model=if_model_pairwise,
    math_model=math_model_pairwise,
    math_to_if_model=math_to_if_pairwise,
)
tau_pairwise = build_tau_map(pass1=pass1_pairwise, tau_rms_factor=TAU_RMS_FACTOR)
conflict_pairwise = collect_ifbase_vs_stage2_conflict_stats(
    base_model=base_model_pairwise,
    if_model=if_model_pairwise,
    math_model=math_model_pairwise,
    math_to_if_model=math_to_if_pairwise,
    tau_map=tau_pairwise,
)
pairwise_df = build_layer_dataframe(pass1=pass1_pairwise, conflict=conflict_pairwise, tau_map=tau_pairwise)

# Add lm_head diagnostics from state_dict so tied-weight architectures are still covered.
pairwise_lm_head_row, pairwise_lm_head_debug = compute_ifbase_vs_stage2_lm_head_stats(
    base_model=base_model_pairwise,
    if_model=if_model_pairwise,
    math_model=math_model_pairwise,
    math_to_if_model=math_to_if_pairwise,
    tau_rms_factor=TAU_RMS_FACTOR,
)
if pairwise_lm_head_row is not None:
    pairwise_df = pairwise_df[pairwise_df["layer"] != "layer_lm_head"].copy()
    pairwise_df = pd.concat([pairwise_df, pd.DataFrame([pairwise_lm_head_row])], ignore_index=True)

# Keep final norm excluded to avoid degenerate interpretation for tiny vectors.
pairwise_df = pairwise_df[pairwise_df["layer"] != "layer_final_norm"].copy()
pairwise_df = pairwise_df.sort_values("layer", key=lambda col: col.map(layer_sort_key)).reset_index(drop=True)

# Recompute normalized shares after row filtering/injection.
denom_pairwise = (pairwise_df["norm_if_l"] + pairwise_df["norm_math_l"]).replace(0.0, 1e-12)
pairwise_df["norm_if_share"] = pairwise_df["norm_if_l"] / denom_pairwise
pairwise_df["norm_math_share"] = pairwise_df["norm_math_l"] / denom_pairwise

# Semantic aliases make the vector meanings explicit in CSV outputs.
pairwise_df["norm_if_minus_base_l"] = pairwise_df["norm_if_l"]
pairwise_df["norm_math_to_if_minus_math_l"] = pairwise_df["norm_math_l"]
pairwise_df["share_if_minus_base"] = pairwise_df["norm_if_share"]
pairwise_df["share_math_to_if_minus_math"] = pairwise_df["norm_math_share"]

pairwise_report_df = pairwise_df.copy()
pairwise_report_df["risk_score"] = (
    pairwise_report_df["conflict_l_tau_mean"].rank(pct=True)
    + pairwise_report_df["sign_conflict_ratio_nonzero"].rank(pct=True)
    + (1.0 - pairwise_report_df["cancellation_ratio"]).rank(pct=True)
    + (1.0 - (pairwise_report_df["cos_l"] + 1.0) * 0.5).rank(pct=True)
)

pairwise_csv_path = PAIRWISE_ARTIFACT_DIR / "if_minus_base_vs_math_to_if_minus_math_layer_interference_stats.csv"
pairwise_risk_path = PAIRWISE_ARTIFACT_DIR / "if_minus_base_vs_math_to_if_minus_math_layer_risk_ranking.csv"
pairwise_debug_path = PAIRWISE_ARTIFACT_DIR / "if_minus_base_vs_math_to_if_minus_math_lm_head_debug.json"

pairwise_df.to_csv(pairwise_csv_path, index=False)
pairwise_report_df.sort_values("risk_score", ascending=False).to_csv(pairwise_risk_path, index=False)
with pairwise_debug_path.open("w", encoding="utf-8") as f:
    json.dump(pairwise_lm_head_debug, f, indent=2)

print(f"Saved pairwise stats: {pairwise_csv_path}")
print(f"Saved pairwise risk ranking: {pairwise_risk_path}")
print(f"Saved pairwise lm_head debug: {pairwise_debug_path}")
print(f"Contains layer_lm_head: {'layer_lm_head' in set(pairwise_df['layer'].tolist())}")
print(f"Contains layer_final_norm: {'layer_final_norm' in set(pairwise_df['layer'].tolist())}")

display(pairwise_df)
visualize_layer_interference(df=pairwise_df, output_dir=PAIRWISE_ARTIFACT_DIR)

# Explicit cleanup to avoid GPU memory accumulation across notebook sections.
del base_model_pairwise
del if_model_pairwise
del math_model_pairwise
del math_to_if_pairwise
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
